In [1]:
import sys
from pathlib import Path 
import math
import pickle
import bisect
import numpy as np
import time

from dataclasses import dataclass
import random
from collections import defaultdict

W_SIZE = 1000
INF = 10**18
ONLINE_JUDGE = True if len(sys.argv) >= 2 and sys.argv[1] == "ONLINE_JUDGE" else False

def fastcopy(obj):
    return pickle.loads(pickle.dumps(obj, -1))

def debug_print(*args, **kwargs):
    if not ONLINE_JUDGE:
        print(*args, **kwargs)

In [2]:
class Env:
    def __init__(self, input_txt_path: Path):
        self.W, self.D, self.N, self.a = self._input(input_txt_path)

    def _input(self, txt_path):
        with open(txt_path, mode="r") as file:
            lines = file.readlines()
        W, D, N = map(int, lines[0].split())
        a = []
        for line, d in zip(lines[1:], range(D)):
            d = list(map(int, line.split()))
            a.append(d)
        return W, D, N, a

In [3]:
@dataclass
class Rect:
    x1: int
    y1: int
    x2: int
    y2: int

    def calc_area(self):
        return (self.x2 - self.x1) * (self.y2 - self.y1)

In [4]:
def overlap_1d(xl1: int, xr1: int, xl2: int, xr2: int):
    return True if max(xl1, xl2) < min(xr1, xr2) else False


def overlap_1d_ok_just(xl1: int, xr1: int, xl2: int, xr2: int):
    return True if max(xl1, xl2) <= min(xr1, xr2) else False


def overlap_1d_vec_ok_just(vec1: tuple[int, int], vec2: tuple[int, int]):
    return overlap_1d_ok_just(vec1[0], vec1[1], vec2[0], vec2[1])


def overlap_1d_vec(vec1: tuple[int, int], vec2: tuple[int, int]):
    return overlap_1d(vec1[0], vec1[1], vec2[0], vec2[1])


def get_overlap_vec(vec1: tuple[int, int], vec2: tuple[int, int]):
    return (max(vec1[0], vec2[0]), min(vec1[1], vec2[1]))


def concat_1d_vec(vecs: list[tuple[int, int]]) -> list[tuple[int, int]]:
    serched_vecs = []
    for vec in vecs:
        overlap_vecs = [vec]
        non_overlap_vecs = []
        for seached_vec in serched_vecs:
            if overlap_1d_vec_ok_just(vec, seached_vec):
                overlap_vecs.append(seached_vec)
            else:
                non_overlap_vecs.append(seached_vec)
        concat_overlap_vec = (min([v[0] for v in overlap_vecs]), max([v[1] for v in overlap_vecs]))
        serched_vecs = non_overlap_vecs + [concat_overlap_vec]
    return serched_vecs


def evaluate_1d_partition_costs(pre_vecs: list[tuple[int, int]], next_vecs: list[tuple[int, int]]) -> int:
    pre_vecs = concat_1d_vec(pre_vecs)
    next_vecs = concat_1d_vec(next_vecs)

    overlaped_vecs = []
    for pre_vec in pre_vecs:
        for next_vec in next_vecs:
            if overlap_1d_vec(pre_vec, next_vec):
                overlaped_vecs.append(get_overlap_vec(pre_vec, next_vec))

    cost = 0
    cost += sum(vec[1] - vec[0] for vec in pre_vecs)
    cost += sum(vec[1] - vec[0] for vec in next_vecs)

    converged_overlaped_vecs = concat_1d_vec(overlaped_vecs)
    for vec in converged_overlaped_vecs:
        cost -= (vec[1] - vec[0]) * 2
    return cost


def evaluate_partition_rect_cost(pre_rect: list[Rect], next_rect: list[Rect]) -> int:
    pre_rect_x_vecs = defaultdict(list)
    pre_rect_y_vecs = defaultdict(list)
    next_rect_x_vecs = defaultdict(list)
    next_rect_y_vecs = defaultdict(list)

    for rect in pre_rect:
        pre_rect_x_vecs[rect.y1].append((rect.x1, rect.x2))
        pre_rect_x_vecs[rect.y2].append((rect.x1, rect.x2))
        pre_rect_y_vecs[rect.x1].append((rect.y1, rect.y2))
        pre_rect_y_vecs[rect.x2].append((rect.y1, rect.y2))
    for rect in next_rect:
        next_rect_x_vecs[rect.y1].append((rect.x1, rect.x2))
        next_rect_x_vecs[rect.y2].append((rect.x1, rect.x2))
        next_rect_y_vecs[rect.x1].append((rect.y1, rect.y2))
        next_rect_y_vecs[rect.x2].append((rect.y1, rect.y2))

    cost = 0

    x_keys = set(pre_rect_x_vecs.keys()) | set(next_rect_x_vecs.keys())
    for key in x_keys:
        if 0 < key < W_SIZE:
            cost += evaluate_1d_partition_costs(pre_rect_x_vecs[key], next_rect_x_vecs[key])

    y_keys = set(pre_rect_y_vecs.keys()) | set(next_rect_y_vecs.keys())
    for key in y_keys:
        if 0 < key < W_SIZE:
            cost += evaluate_1d_partition_costs(pre_rect_y_vecs[key], next_rect_y_vecs[key])
    return cost


def evaluate_area_cost(target_rects: list[Rect], target_areas: list[int]) -> int:
    cost = 0
    for i in range(len(target_rects)):
        rect_area = abs(target_rects[i].x2 - target_rects[i].x1) * abs(target_rects[i].y2 - target_rects[i].y1)
        if rect_area < target_areas[i]:
            cost += target_areas[i] - rect_area
    return cost * 100


def evaluate_one_day_cost(pre_rect: list[Rect] | None, next_rect: list[Rect], target_areas: list[int]) -> int:
    if pre_rect is None:
        partition_cost = 0
    else:
        partition_cost = evaluate_partition_rect_cost(pre_rect, next_rect)
    area_cost = evaluate_area_cost(next_rect, target_areas)
    return partition_cost + area_cost


def evaluate_all_cost_from_list(ans: list[list[int]], target_areas: list[list[int]]) -> int:
    all_rects = []
    days = len(ans)
    for day in range(days):
        rects = []
        for vec in ans[day]:
            rects.append(Rect(vec[0], vec[1], vec[2], vec[3]))
        all_rects.append(rects)
    return evaluate_all_cost_from_rect(all_rects, target_areas)

def evaluate_all_cost_from_rect(ans: list[list[Rect]], target_areas: list[list[int]]) -> int:
    cost = 0
    days = len(ans)
    for day in range(days):
        if day == 0:
            cost += evaluate_one_day_cost(None, ans[day], target_areas[day])
        else:
            cost += evaluate_one_day_cost(ans[day - 1], ans[day], target_areas[day])
    return cost + 1

In [5]:
@dataclass
class Node:
    idx: int
    action: tuple[str, float] | None
    rect: Rect | None
    p: int | None
    l: int | None
    r: int | None

In [6]:
def div_rects(rect: Rect, action: tuple[str, float]) -> tuple[Rect, Rect]:
    direct = action[0]
    if rect.x2 - rect.x1 == 1:
        direct = "H"
    if rect.y2 - rect.y1 == 1:
        direct = "V"
    if direct == "V":
        nx = int(rect.x1 + (rect.x2 - rect.x1) * action[1])
        x = min(max(rect.x1 + 1, nx), rect.x2 -1)
        if x == rect.x1 or x == rect.x2:
            return None, None
        return (Rect(rect.x1, rect.y1, x, rect.y2), Rect(x, rect.y1, rect.x2, rect.y2))
    else:
        ny = int(rect.y1 + (rect.y2 - rect.y1) * action[1])
        y = min(max(rect.y1+1, ny), rect.y2 - 1)
        if y == rect.y1 or y == rect.y2:
            return None, None
        return (Rect(rect.x1, rect.y1, rect.x2, y), Rect(rect.x1, y, rect.x2, rect.y2))

In [7]:
def init_action_tmp(N):
    nodes: dict[int, Node] = {}

    node_zero = Node(
        idx=0,
        action=None,
        p=None,
        l=None,
        r=None,
        rect=Rect(0, 0, W_SIZE, W_SIZE),
    )

    nodes[0] = node_zero
    child_nodes = [node_zero]

    for i in range(1, N):
        child_node = random.choice(child_nodes)
        idx_left = i*2-1
        idx_right = i*2
        if child_node.rect.x2 - child_node.rect.x1 < child_node.rect.y2 - child_node.rect.y1:
            random_direct = "H"
        else:
            random_direct = "V"
        random_position = (0.2 - random.random() / 10.0) + 0.2
        random_action = (random_direct, random_position)

        child_node.l = idx_left
        child_node.r = idx_right
        child_node.action = random_action
        nodes[child_node.idx] = child_node

        rect_left, rect_right = div_rects(child_node.rect, random_action)
        if rect_left is None:
            return []

        node_left = Node(
            idx=idx_left,
            action=None,
            p=child_node.idx,
            l=None,
            r=None,
            rect=rect_left,
        )
        node_right = Node(
            idx=idx_right,
            action=None,
            p=child_node.idx,
            l=None,
            r=None,
            rect=rect_right,
        )
        nodes[idx_left] = node_left
        nodes[idx_right] = node_right

        child_nodes.remove(child_node)
        child_nodes.append(node_left)
        child_nodes.append(node_right)
    return nodes

def init_action(N):
    while True:
        nodes = init_action_tmp(N)
        if len(nodes) > 0:
            return nodes

In [8]:
from collections import deque
def update_node_rects(nodes: dict[int, Node]):

    child_node_idxs = deque([0])
    while child_node_idxs:
        now_idx = child_node_idxs.pop()
        if nodes[now_idx].action is None:
            continue
        else:
            rect_left, rect_right = div_rects(nodes[now_idx].rect, nodes[now_idx].action)
            if rect_left is None:
                return False
            left_idx = nodes[now_idx].l
            right_idx = nodes[now_idx].r
            nodes[left_idx].rect = rect_left
            nodes[right_idx].rect = rect_right
            child_node_idxs.append(left_idx)
            child_node_idxs.append(right_idx)
    return True

In [9]:
def random_range(a, b):
    return a + (b - a) * random.random()

In [10]:
def clac_having_nodes(nodes: dict[int, Node]):
    node_idxs_q = deque([0])
    node_dfs = []
    while node_idxs_q:
        now_idx = node_idxs_q.popleft()
        node_dfs.append(now_idx)
        if nodes[now_idx].l is not None:
            node_idxs_q.append(nodes[now_idx].l)
            node_idxs_q.append(nodes[now_idx].r)
    node_dfs = deque(node_dfs)
    
    having_nodes = {}
    for node in nodes.values():
        having_nodes[node.idx] = 0

    while node_dfs:
        now_idx = node_dfs.pop()
        p_idx = nodes[now_idx].p
        if p_idx is None:
            continue
        having_nodes[p_idx] += having_nodes[now_idx] + 1
    
    return having_nodes

In [11]:
def get_child_node_idxs(nodes: dict[int, Node], idx: int):
    child_nodes = []
    if nodes[idx].l is not None:
        child_nodes.append(nodes[idx].l)
        child_nodes.append(nodes[idx].r)
    return child_nodes

In [12]:
EPS = 1e-9
MAX_SLIDE = 30

def change_nearby_nodes(nodes: dict[int, Node]):
    ret_nodes = fastcopy(nodes)
    act_num = random.choice([0, 1, 2])
    
    if act_num == 0:
        # ノードの分割割合を変更
        having_action_idxs = []
        for i in range(len(nodes)):
            if nodes[i].action is not None:
                having_action_idxs.append(i)
        target_idx = random.choice(having_action_idxs)
        target_node = nodes[target_idx]
        target_rect = target_node.rect
        direct, position = target_node.action
        if random.random() < 0.5:
            pm = 1
        else:
            pm = -1
        if direct == "V":
            w = target_rect.x2 - target_rect.x1
            random_diff = pm * random_range(1.0/w + EPS, MAX_SLIDE/w)
        else:
            h = target_rect.y2 - target_rect.y1
            random_diff = pm * random_range(1.0/h + EPS, MAX_SLIDE/h)
        position = min(max(0.0, position + random_diff), 1.0)
        target_node.action = (direct, position)
        ret_nodes[target_idx] = target_node
    elif act_num == 1:
        # ランダムに分割方向を変更
        having_nodes = clac_having_nodes(nodes)
        having_action_idxs = []
        for i in range(len(nodes)):
            if nodes[i].action is not None and having_nodes[i] == 2:
                having_action_idxs.append(i)
        target_idx = random.choice(having_action_idxs)
        target_node = nodes[target_idx]
        direct, position = target_node.action
        if direct == "V":
            target_node.action = ("H", position)
        else:
            target_node.action = ("V", position)
        ret_nodes[target_idx] = target_node
    elif act_num == 2:
        having_nodes = clac_having_nodes(nodes)
        random_select_target_idxs = []
        having_zero_nodes = []
        for node_idx, having_num in having_nodes.items():
            if node_idx == 0:
                continue
            if having_num == 0:
                having_zero_nodes.append(node_idx)
            elif 2 <= having_num <= 2:
                random_select_target_idxs.append(node_idx)
        # print(random_select_target_idxs, having_zero_nodes)
        # print(having_nodes)
        target_idx = random.choice(random_select_target_idxs)
        target_node_childs = get_child_node_idxs(nodes, target_idx)
        having_zero_nodes = list(set(having_zero_nodes) - set(target_node_childs))
        target_child_idx = random.choice(having_zero_nodes)
        nodes[target_child_idx].l = nodes[target_idx].l
        nodes[target_child_idx].r = nodes[target_idx].r
        nodes[target_child_idx].action = nodes[target_idx].action
        nodes[nodes[target_idx].l].p = target_child_idx
        nodes[nodes[target_idx].r].p = target_child_idx
        nodes[target_idx].l = None
        nodes[target_idx].r = None
        nodes[target_idx].action = None
    else:
        raise ValueError("act_num is invalid")

    if not update_node_rects(ret_nodes):
        return nodes

    return ret_nodes

In [13]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def display_map(rects: list[Rect]):
    fig = plt.figure(figsize=(4, 4))
    ax = plt.axes()

    r = patches.Rectangle(xy=(0, 0), width=1000, height=1000, ec='#000000', fill=False)
    ax.add_patch(r)

    for rect in rects:
        width = rect.x2 - rect.x1
        height = rect.y2 - rect.y1
        r = patches.Rectangle(xy=(rect.x1, rect.y1), width=width, height=height, ec='#000000', fill=True)
        ax.add_patch(r)

    plt.xlim(-5, 1005)
    plt.ylim(-5, 1005)
    plt.axis('off')
    ax.set_aspect('equal')
    plt.show()

In [14]:
def simulate_anneling(target_areas, day, pre_nodes: dict[int, Node] | None):
    days = len(target_areas)
    N = len(target_areas[0])

    if pre_nodes:
        x_best = pre_nodes
        pre_rects = [node.rect for node in pre_nodes.values() if node.l is None]
    else:        
        x_best = init_action(N)
        pre_rects = None
    now_rects = [node.rect for node in x_best.values() if node.l is None]
    now_rects = sorted(now_rects, key=lambda x: x.calc_area())
    fx_best = evaluate_one_day_cost(pre_rects, now_rects, target_areas[day])
    
    SIMULATE_N = 1000000
    for i in range(SIMULATE_N):
        x = change_nearby_nodes(x_best)
        now_rects = [node.rect for node in x.values() if node.l is None]
        now_rects = sorted(now_rects, key=lambda x: x.calc_area())
        fx = evaluate_one_day_cost(pre_rects, now_rects, target_areas[day])
        next_cost = 0
        for d in range(day+1, days):
            weight = (1/4) ** (day+1 - d)
            next_cost += evaluate_area_cost(now_rects, target_areas[d]) * (1- i/SIMULATE_N) * weight * 0.0001
        fx += next_cost
        if fx < fx_best:
            x_best = x
            fx_best = fx
        if i % 5000 == 0:
            print(f"{i}/{SIMULATE_N} {fx_best} {next_cost}")

    best_rects = [node.rect for node in x_best.values() if node.l is None]
    best_rects = sorted(best_rects, key=lambda x: x.calc_area())
    return x_best, best_rects

In [15]:
def solve(env: Env):
    ans = []
    x_best = None
    for d in range(env.D):
        print(f"day {d}")
        x_best, best_rects = simulate_anneling(env.a, d, x_best)
        ans.append(best_rects)
    return ans

In [16]:
i = 0

debug_print(f"----- {i}/100 -----")
input_txt_path = Path(f"./in/{str(i).zfill(4)}.txt")
output_txt_path = Path(f"./out/{str(i).zfill(4)}.txt")
env = Env(input_txt_path)

ans = solve(env)
cost = evaluate_all_cost_from_rect(ans, env.a)
debug_print(f"cost:{cost}")

str_ans = []
for d in range(env.D):
    for k in range(env.N):
        r: Rect = ans[d][k]
        i0, j0, i1, j1 = r.x1, r.y1, r.x2, r.y2
        str_ans.append(f"{i0} {j0} {i1} {j1}")

with open(output_txt_path, mode="w") as file:
    file.write("\n".join(str_ans))

----- 0/100 -----
day 0
0/1000000 4498700 96765.88
5000/1000000 0.0 278472.8987
10000/1000000 0.0 254632.4154
15000/1000000 0.0 482576.4993
20000/1000000 0.0 305724.9454
25000/1000000 0.0 169800.70575000002
30000/1000000 0.0 437349.26410000003
35000/1000000 0.0 339869.82515
40000/1000000 0.0 191865.2256
45000/1000000 0.0 244314.41255
50000/1000000 0.0 368227.2105
55000/1000000 0.0 189351.6912
60000/1000000 0.0 349403.4426
65000/1000000 0.0 218623.9253
70000/1000000 0.0 170951.3445
75000/1000000 0.0 298437.18075000006
80000/1000000 0.0 167907.7832
85000/1000000 0.0 307389.62925
90000/1000000 0.0 349287.6933
95000/1000000 0.0 335407.56145000004
100000/1000000 0.0 177957.531
105000/1000000 0.0 151113.948
110000/1000000 0.0 325410.8335
115000/1000000 0.0 227811.75285000002
120000/1000000 0.0 173496.85760000002
125000/1000000 0.0 321487.94125000003
130000/1000000 0.0 205125.44190000003
135000/1000000 0.0 96149.87945000001
140000/1000000 0.0 397348.06179999997
145000/1000000 0.0 288957.98025

KeyboardInterrupt: 

In [ ]:
# env.N = 20
# best_x = init_action(env.N)
# rects = [node.rect for node in best_x.values() if node.l is None]
# best_fx = evaluate_one_day_cost(None, rects, env.a[0])

# for i in range(10000):
#     best_x = change_nearby_nodes(best_x)
#     rects = [node.rect for node in best_x.values() if node.l is None]
#     if i % 1000 == 0:
#         print(i)
#         display_map(rects)